**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**13 | Final Development Training and Research Evidence Synthesis**

![HARTH final Development fitting and research evidence workflow](../../images/13_consolidated_workflow.svg)

**Objective:** Fit the Development-selected weighted 1D-ResNet and fixed Random Forest reference using the established HARTH v2.0 procedure, save reusable inference artefacts and synthesise the scientific evidence from the current study.

**Source:** HARTH v2.0 at the pinned revision, with six acceleration channels, eight activity codes and 29 Development participants (14,201 label-pure windows under a 500-window-per-person cap). [Notebook 12](12_harth_development_model_selection_and_protocol_specification.ipynb) defines model selection and fitting settings; [Notebooks 10](10_harth_advanced_temporal_architectures_and_matched_benchmark.ipynb) and [11](11_harth_multiseed_stability_and_imbalance_robustness.ipynb) contain the matched Development and robustness comparisons.

**Scientific boundary:** The reserved participants S023 and S032 are excluded from all fitting and new inference. Saved predictions from the earlier qualified, same-study held-out evaluation can be summarised if their genuine aligned CSV is available. Historical Test scores are not attributed to the newly fitted models. No external, blind or clinical validation is claimed.

**Outputs:** Development cohort, class support, fitting settings, ResNet training loss, model archive and reload check, historical qualified evidence where available, and a concise research interpretation.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ---------------------------- 2. Standard Library ----------------------------
import json
from zipfile import ZipFile, ZIP_DEFLATED

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# -------------------- 4. Machine Learning and Deep Learning ------------------
import joblib
import torch
from sklearn.ensemble import RandomForestClassifier
from core.computer_device import computer_device
from core.harth_final_inference import (
    fit_final_resnet, load_final_models, infer_source_windows,
)
from core.harth_qualified_test_report import (
    summarise_qualified_heldout, SOURCE_CLASSES, RESERVED_PEOPLE,
)
from core.feature_engineering import extract_signal_features
from harth_data import (
    HARTH_REVISION, HARTH_SUBJECTS, SENSOR_COLUMNS, ACTIVITY_NAMES,
    prepare_harth_window_cohort,
)

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# -------------------------- 6. Outputs and Compute Device --------------------
from core.output_download import download_outputs_from_colab
output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
model_dir = root / "data" / "processed" / "harth_final_models"
model_dir.mkdir(parents=True, exist_ok=True)
raw_dir = root / "data" / "raw" / "harth_v2"
CLASSES = tuple(SOURCE_CLASSES)
RESERVED = set(RESERVED_PEOPLE)
DEVICE = str(computer_device(min_vram_gb=None))
print("Final Development device:", DEVICE)
print("Study outputs:", output_dir)

**02. Read Development Model Selection and Fitting Settings**

Load the current `12_` configuration tables and the original `10_`/`11_` Development evidence. This stage retains the 400-update weighted ResNet, 120-tree Random Forest and fixed-eight-class Macro F1 reporting definition.

The files belong to the earlier, completed Development analysis. Their contents specify the fitting procedure rather than providing predictions for the final model.

In [ ]:
# =============== 02. Read Model Selection and Fitting Settings ===============

# ----------------------- 1. Current Required Input Files ---------------------
input_files = {
    "selection": "12_development_model_selection_decisions.csv",
    "source": "12_recorded_source_protocol.csv",
    "training": "12_recorded_training_configuration.csv",
    "scope": "12_qualified_evaluation_scope.csv",
    "benchmark": "10_matched_cv_model_summary.csv",
    "robustness": "11_robustness_model_summary.csv",
}
absent = [name for name in input_files.values()
          if not (output_dir / name).is_file()]
if absent:
    raise FileNotFoundError(
        "Publish Notebook 12 CSV outputs before running Notebook 13: "
        + ", ".join(absent)
    )
tables = {key: pd.read_csv(output_dir / name)
          for key, name in input_files.items()}
source = tables["source"].set_index("property")["recorded_value"]
training = tables["training"].set_index("setting")["recorded_value"]
scope = tables["scope"].set_index("item")["recorded_rule"]

# ----------------------- 2. Check the Selected Recipe ------------------------
if (
    str(training["Primary architecture"]) != "1D-ResNet"
    or str(training["Reference"]) != "Random Forest"
    or str(training["Loss"]) != "inverse_sqrt_weighted_ce"
    or str(training["Optimiser"]) != "Adam"
    or int(training["Optimiser updates"]) != 400
    or int(training["Batch size"]) != 128
    or float(training["Learning rate"]) != 0.001
    or int(training["RF trees"]) != 120
    or int(training["RF minimum leaf"]) != 3
    or str(training["RF class weighting"]) != "balanced_subsample"
    or str(scope["Primary metric"]) != "Fixed-eight-class Macro F1"
):
    raise ValueError("Notebook 12 selected configuration differs.")
if str(source["Pinned revision"]) != HARTH_REVISION:
    raise ValueError("Pinned HARTH source revision differs.")
if str(source["Reserved participants"]) != "S023, S032":
    raise ValueError("Unexpected reserved participant assignments.")
if int(source["Window size (samples)"]) != 100 or int(
    source["Stride (samples)"]
) != 50:
    raise ValueError("Unexpected source window specification.")
protocol = {"primary_model": {
    "architecture": "1D-ResNet",
    "classes": list(CLASSES),
    "random_seed": int(training["Seed"]),
    "batch_size": int(training["Batch size"]),
    "learning_rate": float(training["Learning rate"]),
    "optimizer_steps": int(training["Optimiser updates"]),
}}
display(tables["selection"][["role", "configuration", "rationale"]])

**03. Reconstruct the Development Cohort**

Read only the original 29 Development participants from the pinned HARTH source. The data adapter applies time-continuous, fully labelled windows before excluding activities outside codes 1–8. The 500-window cap is uniform in source order, not chosen by activity.

Source identities and row alignment are verified before fitting; overlapping windows remain correlated, so independent evaluation must be defined at participant level.

In [ ]:
# ================== 03. Reconstruct the Development Cohort ===================

# -------------------- 1. Separate Participant Identities ---------------------
development_people = sorted(set(HARTH_SUBJECTS) - RESERVED)
if len(development_people) != 29 or RESERVED & set(development_people):
    raise ValueError("Unexpected Development participant split.")

# --------------------- 2. Prepare Original Source Windows --------------------
X_dev, y_dev, metadata, cohort = prepare_harth_window_cohort(
    subjects=development_people, data_dir=raw_dir,
    task_classes=CLASSES,
    max_windows_per_subject=int(source["Development cap per person"]),
    window_size=int(source["Window size (samples)"]),
    step_size=int(source["Stride (samples)"]),
    sampling_rate=int(source["Sampling rate (Hz)"]),
)
if (X_dev.ndim != 3 or X_dev.shape[1:] != (6, 100)
        or len(X_dev) != len(y_dev) or len(metadata) != len(y_dev)):
    raise ValueError("Development window/source alignment failed.")
if set(metadata["subject_id"]) != set(development_people):
    raise ValueError("Incomplete or overlapping participant coverage.")
if not np.array_equal(y_dev, metadata["label"].to_numpy()):
    raise ValueError("Target labels are not aligned with signal windows.")
if not np.isin(y_dev, CLASSES).all() or not np.isfinite(X_dev).all():
    raise ValueError("Invalid input activity codes or sensor values.")
if len(X_dev) != int(source["Development windows"]):
    raise ValueError("Original Development selection differs from Notebook 12.")

# ------------------------ 3. Save Source-Level Support -----------------------
cohort.to_csv(output_dir / "13_development_cohort_summary.csv",
              index=False)
support = pd.DataFrame({
    "source_code": list(CLASSES),
    "activity": [ACTIVITY_NAMES[c] for c in CLASSES],
    "development_windows": [
        int(np.count_nonzero(y_dev == c)) for c in CLASSES
    ],
})
support.to_csv(output_dir / "13_development_activity_support.csv",
               index=False)
display(support)
print("Development participants:", len(development_people))
print("Development windows:", len(X_dev))

**04. Train the Final 1D-ResNet**

Use the previously selected 1D-ResNet, inverse-square-root weighted Cross-Entropy, Adam learning rate 0.001, batch size 128, 400 optimiser updates and seed 42. Channel standardisation and class weights are estimated exclusively from Development data.

The fixed final-step checkpoint is not selected using held-out results. Loss values describe training optimisation, not independent generalisation.

In [ ]:
# ================== 04. Train the Final Weighted 1D-ResNet ===================

# -------------------- 1. Fit the Established Configuration -------------------
model, scaler, train_history, class_weights, fit_seconds = (
    fit_final_resnet(X_dev, y_dev, protocol, device=DEVICE)
)
if (train_history.empty or
        not np.isfinite(train_history["development_train_loss"]).all()):
    raise ValueError("Invalid Development training trace.")
if int(train_history["step"].iloc[-1]) != 400:
    raise ValueError("Final-step training record is incomplete.")

# --------------------- 2. Save Training and Scaler Evidence ------------------
train_history.to_csv(
    output_dir / "13_resnet_training_history.csv", index=False
)
pd.DataFrame({
    "channel": SENSOR_COLUMNS, "development_mean": scaler["mean"],
    "development_std": scaler["std"],
}).to_csv(output_dir / "13_development_channel_scaling.csv",
          index=False)
pd.DataFrame({
    "source_code": list(CLASSES),
    "activity": [ACTIVITY_NAMES[c] for c in CLASSES],
    "development_class_weight": np.asarray(class_weights).reshape(-1),
}).to_csv(output_dir / "13_resnet_class_weights.csv", index=False)

# ------------------------ 3. Plot Development Loss ---------------------------
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_history["step"],
        train_history["development_train_loss"], marker="o")
ax.set(xlabel="Adam optimiser update",
       ylabel="Development training cross-entropy",
       title="Final weighted 1D-ResNet | Training loss")
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "13_resnet_training_loss.png")
plt.show()
display(train_history)
print("Neural fitting time (seconds):", round(fit_seconds, 2))

**05. Fit the Fixed Random Forest Reference**

Use 30 deterministic features (mean, SD, RMS, peak-to-peak amplitude and waveform length for six channels) from the same eligible Development windows. Fit 120 trees, minimum leaf size three, balanced-subsample class weights and seed 42.

This is a configured-pipeline comparison: the classical model uses engineered features whereas the ResNet uses standardised signal sequences.

In [ ]:
# ==================== 05. Fit the Classical Random Forest ====================

# ---------------------- 1. Original-Unit Feature Matrix ----------------------
features = extract_signal_features(
    X_dev, channel_names=SENSOR_COLUMNS
)
if (features.shape != (len(X_dev), 30) or
        not np.isfinite(features.to_numpy()).all()):
    raise ValueError("Invalid Random Forest features.")

# ---------------------- 2. Fit the Classical Reference -----------------------
rf = RandomForestClassifier(
    n_estimators=int(training["RF trees"]),
    min_samples_leaf=int(training["RF minimum leaf"]),
    class_weight=str(training["RF class weighting"]),
    random_state=int(training["RF seed"]),
    n_jobs=-1,
)
rf.fit(features.to_numpy(), y_dev)
if set(rf.classes_) != set(CLASSES):
    raise ValueError("Missing fitted reference activity classes.")
print("Random Forest trees:", len(rf.estimators_))
print("Statistical features:", rf.n_features_in_)

**06. Save and Reload Reusable Model Artefacts**

Save the final ResNet weights and fitted Random Forest, along with the minimal inference configuration containing the Development-only channel statistics, activity mapping and feature-column order. This configuration is necessary to reproduce inference; it is not a historical approval file.

Reload the locally generated artefacts and compare the reference-model predictions on a small Development sample. Do not load untrusted joblib files. Preserve a durable copy of the exported archive because Colab storage is ephemeral.

In [ ]:
# =================== 06. Save and Reload the Final Models ====================

# ---------------------- 1. Save Models and Configuration ---------------------
net_file = model_dir / "resnet_state_dict.pt"
forest_file = model_dir / "random_forest.joblib"
config_file = model_dir / "inference_config.json"
torch.save(
    {key: value.detach().cpu()
     for key, value in model.state_dict().items()},
    net_file,
)
joblib.dump(rf, forest_file)
config = {
    "source_dataset": "HARTH v2.0",
    "source_revision": HARTH_REVISION,
    "sensor_columns": list(SENSOR_COLUMNS),
    "source_codes": list(CLASSES),
    "channel_mean": scaler["mean"].tolist(),
    "channel_std": scaler["std"].tolist(),
    "feature_columns": list(features.columns),
    "resnet_architecture": "1D-ResNet",
    "resnet_hidden_channels": 32,
    "window_size": 100,
    "window_stride": 50,
    "nominal_hz": 50,
    "preprocessing_fit_scope": "Development only",
    "test_fitted_values": False,
    "optimiser_updates": 400,
    "training_seed": int(training["Seed"]),
}
config_file.write_text(
    json.dumps(config, indent=2) + "\n", encoding="utf-8"
)

# -------------------- 2. Reload and Validate Inference -----------------------
loaded_net, loaded_rf, reloaded_config = load_final_models(
    model_dir, device="cpu"
)
if reloaded_config != config:
    raise ValueError("Inference configuration changed during reload.")
if list(loaded_rf.classes_) != list(CLASSES):
    raise ValueError("The reloaded model has a different class order.")

# -------------------- 3. Development-Only Reload Check -----------------------
n_check = min(24, len(y_dev))
pred_resnet, pred_rf = infer_source_windows(
    X_dev[:n_check], loaded_net, loaded_rf, reloaded_config,
    device="cpu",
)
np.testing.assert_array_equal(
    pred_rf, rf.predict(features.iloc[:n_check].to_numpy())
)
if not set(pred_resnet).issubset(CLASSES):
    raise ValueError("Reloaded ResNet returned invalid class codes.")
dev_check = pd.DataFrame({
    "subject_id": metadata["subject_id"].iloc[:n_check].to_numpy(),
    "true_development_code": y_dev[:n_check],
    "reloaded_resnet_prediction": pred_resnet,
    "reloaded_rf_prediction": pred_rf,
})
dev_check.to_csv(
    output_dir / "13_development_inference_example.csv", index=False
)
display(dev_check.head(10))

**07. Record the Final Fitting Procedure**

Document the observed Development cohort and the exact fitting settings. Training time and a successful reload are engineering observations, not independent measures of predictive performance. The newly fitted models do not inherit the historically recorded Test performance.

In [ ]:
# ================== 07. Summarise Final Development Fitting ==================

# ------------------------ 1. Save Current Run Settings -----------------------
training_summary = pd.DataFrame([
    ("HARTH source revision", HARTH_REVISION),
    ("Development participants", len(development_people)),
    ("Development windows", len(X_dev)),
    ("Activity classes", len(CLASSES)),
    ("ResNet loss", "inverse_sqrt_weighted_ce"),
    ("ResNet hidden channels", 32),
    ("ResNet optimiser", "Adam"),
    ("ResNet learning rate", float(training["Learning rate"])),
    ("ResNet batch size", int(training["Batch size"])),
    ("ResNet optimiser updates", 400),
    ("ResNet seed", int(training["Seed"])),
    ("Checkpoint", "final_step"),
    ("Random Forest trees", len(rf.estimators_)),
    ("Random Forest features", rf.n_features_in_),
    ("Neural fitting time (s)", round(fit_seconds, 3)),
    ("Reserved participants used for fitting", False),
], columns=["setting", "recorded_value"])
training_summary.to_csv(
    output_dir / "13_final_training_configuration.csv", index=False
)
display(training_summary)

**08. Summarise Authentic Historical Qualified Evaluation Evidence**

The previous study evaluated models on saved source-aligned windows from S023 and S032; these participants had been inspected descriptively earlier. Its results are therefore qualified same-study held-out evidence, not blind external validation.

If `25_qualified_aligned_predictions.csv` is available in the current study output directory, recompute its original metrics without reloading Test recordings, adapting models or fitting anything. These scores belong only to the historical fitted models. If the original aligned predictions are not available, omit the Test scores rather than infer values from notebook displays.

In [ ]:
# ============== 08. Review Saved Qualified Evaluation Evidence ===============

# ---------------------- 1. Read Original Prediction Rows ---------------------
historical_path = output_dir / "25_qualified_aligned_predictions.csv"
historical_report = None
if historical_path.is_file():
    saved = pd.read_csv(historical_path)
    expected = {
        "subject_id", "true_source_code",
        "resnet_predicted_source_code",
        "random_forest_predicted_source_code",
    }
    if saved.empty or expected.difference(saved.columns):
        raise ValueError("Missing historical aligned prediction fields.")
    historical_preds = {
        "1D-ResNet Weighted CE":
            saved["resnet_predicted_source_code"].to_numpy(),
        "Random Forest":
            saved["random_forest_predicted_source_code"].to_numpy(),
    }

    # ----------------------- 2. Recalculate Saved Metrics --------------------
    historical_report = summarise_qualified_heldout(
        saved["true_source_code"].to_numpy(),
        saved["subject_id"].to_numpy(),
        historical_preds, ACTIVITY_NAMES, CLASSES,
    )
    for key, suffix in [
        ("summary", "model_summary"),
        ("per_subject", "per_subject"),
        ("per_class", "per_class"),
    ]:
        historical_report[key].to_csv(
            output_dir / ("13_historical_qualified_" + suffix + ".csv"),
            index=False,
        )
    display(historical_report["summary"].round(4))
    display(historical_report["per_subject"].round(4))
    print("Historical results recalculated from saved predictions.")
    print("The newly fitted Notebook 13 models were not Test-scored.")
else:
    print(
        "No original aligned qualified-Test CSV is available. "
        "No historical held-out metrics are reported."
    )

**09. Visualise Historical Class-Level Evidence**

When original saved predictions are available, show an eight-class confusion matrix for each historical model, labelled as qualified, same-study evidence. Count matrices and class support must not be mistaken for independent participant observations.

Plots are not fabricated when the original aligned prediction table is unavailable.

In [ ]:
# ================ 09. Visualise Historical Qualified Evidence ================

# ------------------- 1. Plot Saved Eight-Class Counts ------------------------
if historical_report is not None:
    for model_name, cm in historical_report["confusion"].items():
        short = "resnet" if model_name.startswith("1D-") else "random_forest"
        fig, ax = plt.subplots(figsize=(8, 7))
        image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
        ax.set_xticks(range(8), list(map(str, CLASSES)))
        ax.set_yticks(range(8), list(map(str, CLASSES)))
        ax.set(
            xlabel="Predicted source code",
            ylabel="True source code",
            title="Historical qualified confusion | " + model_name,
        )
        for row in range(8):
            for column in range(8):
                ax.text(
                    column, row, str(int(cm[row, column])),
                    ha="center", va="center", fontsize=8,
                    color=("white" if cm[row, column] >
                           0.5 * cm.max() else "black"),
                )
        fig.colorbar(image, ax=ax, label="Window count")
        fig.tight_layout()
        save_figure(
            fig, output_dir /
            f"13_historical_qualified_confusion_{short}.png"
        )
        plt.show()
    subset = historical_report["per_class"].loc[
        lambda frame: frame["model"].eq("1D-ResNet Weighted CE"),
        ["source_code", "activity", "support"],
    ]
    display(subset)
else:
    print("No historical confusion figure without original predictions.")

**10. Interpret Development and Cross-Stage Evidence**

The seven-model benchmark and three-seed robustness comparison establish the Development rationale for selecting weighted ResNet and retaining Random Forest as a classical reference. Sampling diagnostics in Notebook 09 show minority-activity limitations. Calibration analysis in Notebook 08 reports improved mean ECE and Brier score but worsened mean NLL after temperature scaling; this separate experiment was not used to calibrate the final model.

All final claims remain bounded by correlated windows, participant count, class coverage and the absence of independent external or prospective testing. Positive and negative findings are both retained.

In [ ]:
# ==================== 10. Synthesise Scientific Evidence =====================

# ---------------- 1. Display Existing Development Comparisons ----------------
display(tables["benchmark"].sort_values(
    "macro_f1_mean", ascending=False
)[["model", "macro_f1_mean", "macro_f1_sd"]].round(4))
display(tables["robustness"].sort_values(
    "mean_across_seed_fold_macro_f1", ascending=False
)[[
    "architecture", "loss_method",
    "mean_across_seed_fold_macro_f1", "across_seed_sd"
]].round(4))

# -------------------- 2. Distinguish Scientific Claim Levels -----------------
cal_file = output_dir / "08_calibration_mean_metrics.csv"
if cal_file.is_file():
    display(pd.read_csv(cal_file))
evidence_summary = pd.DataFrame([
    ("Development comparison",
     "Seven matched models and three-seed robustness",
     "Descriptive participant-grouped comparison"),
    ("Final model fitting",
     "Weighted ResNet and Random Forest fitted on Development",
     "Training and inference artefacts; no new Test score"),
    ("Rare-class behaviour",
     "Source support and Shuffling precision/recall contrasts",
     "Small class support and overlapping windows"),
    ("Calibration",
     "Mixed Development ECE/Brier/NLL response",
     "Not used for final fitted ResNet"),
    ("Historical qualified assessment",
     ("Aligned saved predictions available"
      if historical_report is not None else
      "No aligned saved predictions supplied"),
     "Two previously inspected participants; not blind external Test"),
    ("Broader applicability",
     "Independent validation is outside this evaluation",
     "No clinical deployment or population-wide claim"),
], columns=["topic", "evidence", "interpretation"])
evidence_summary.to_csv(
    output_dir / "13_research_evidence_summary.csv", index=False
)
display(evidence_summary)

**11. Export Final Model and Research Outputs**

Package the exact ResNet weights, Random Forest estimator and Development-fitted inference configuration from this run. The `13_` output tables and figures describe the final training procedure and its bounded evidence. Historical qualified metrics, when present, remain explicitly distinct from the new model artefacts.

Keep a durable copy of the model ZIP outside Colab. Output tables and figures are saved in the study-level `output/` folder using stable, stage-owned filenames.

In [ ]:
# =================== 11. Export Final Models and Evidence ====================

# ------------------------- 1. Archive Model Artefacts ------------------------
archive_path = output_dir / "13_model_artifacts.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in (
        "resnet_state_dict.pt", "random_forest.joblib",
        "inference_config.json",
    ):
        artifact = model_dir / name
        if not artifact.is_file():
            raise FileNotFoundError("Missing trained model artefact: " + name)
        archive.write(artifact, arcname=name)

# ----------------------- 2. Inventory Current Outputs ------------------------
owned_outputs = sorted(
    file.name for file in output_dir.iterdir()
    if file.is_file() and file.name.startswith("13_")
)
print("Final Development participants:", len(development_people))
print("Final Development windows:", len(y_dev))
print("Model ZIP:", archive_path)
print("Historical Test results:", (
    "summarised from authentic saved predictions"
    if historical_report is not None else "not reported"
))
print("Output files:", len(owned_outputs))
for filename in owned_outputs:
    print(" ", filename)

# ------------------------- 3. Package Colab Exports --------------------------
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="13"
)